# Data Sourcing

Download routine hourly weather observations for Raleigh-Durham International Airport (RDU) from NOAA's Global Historical Climatology Network hourly archive. Times are converted from UTC to US Eastern time before the data are split into local calendar years.

Source: [NOAA NCEI GHCNh](https://www.ncei.noaa.gov/products/global-historical-climatology-network-hourly). Station: `USW00013722`. Field definitions and quality flags are described in the [GHCNh documentation](https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/doc/ghcnh_DOCUMENTATION.pdf).

The target `temperature` is in degrees Celsius. `DATE` and the original `Year`, `Month`, `Day`, and `Hour` fields use UTC. `DATE_local` is an Eastern-time display string without a UTC offset; use timezone-aware timestamps for analysis and UTC for record uniqueness.

In [1]:
import csv
import io
import urllib.request
from collections import defaultdict
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

from tqdm.auto import tqdm

## Settings

`END` is exclusive, so the dates below include all observations through September 16, 2026. The project root is found dynamically so this notebook also works when the repository is stored elsewhere.

In [2]:
NY = ZoneInfo("America/New_York")
START = date(2015, 1, 1)
END = date(2026, 9, 17)
STATION_ID = "USW00013722"  # Raleigh-Durham International Airport

current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    folder
    for folder in [current_dir, *current_dir.parents]
    if (folder / "notebooks").is_dir() and (folder / "requirements.txt").is_file()
)
OUT_DIR = PROJECT_ROOT / "data" / "raw"

BASE_URL = (
    "https://www.ncei.noaa.gov/oa/global-historical-climatology-network/"
    "hourly/access/by-year/{year}/psv/GHCNh_{station}_{year}.psv"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Output folder: {OUT_DIR}")

Project root: /Users/shenweizhang/Desktop/AIPI 520/AIPI 520 Project 1
Output folder: /Users/shenweizhang/Desktop/AIPI 520/AIPI 520 Project 1/data/raw


## Download and prepare the observations

NOAA stores the source files by UTC year. Each observation is converted to Eastern time, filtered to the requested date range, and then grouped by its local year. Only routine hourly reports (`FM15`) recorded at minute 51 are kept.

In [3]:
start_time = datetime(START.year, START.month, START.day, tzinfo=NY)
end_time = datetime(END.year, END.month, END.day, tzinfo=NY)

rows_by_year = defaultdict(list)
column_order = []
columns_with_data = set()

source_years = range(START.year, END.year + 1)
for source_year in tqdm(source_years, desc="Downloading NOAA files", unit="year"):
    url = BASE_URL.format(year=source_year, station=STATION_ID)

    with urllib.request.urlopen(url, timeout=300) as response:
        reader = csv.DictReader(
            io.TextIOWrapper(response, encoding="utf-8", newline=""),
            delimiter="|",
        )
        if not reader.fieldnames:
            raise RuntimeError(f"No header found in {url}")

        for column in reader.fieldnames:
            if column not in column_order:
                column_order.append(column)

        for row in reader:
            if row.get("temperature_Report_Type") != "FM15":
                continue
            if row.get("Minute") != "51":
                continue

            utc_time = datetime.fromisoformat(row["DATE"]).replace(tzinfo=timezone.utc)
            local_time = utc_time.astimezone(NY)
            if not start_time <= local_time < end_time:
                continue

            row["DATE_local"] = local_time.strftime("%Y-%m-%d %H:%M")
            nonempty_row = {name: value for name, value in row.items() if value != ""}
            rows_by_year[local_time.year].append(nonempty_row)
            columns_with_data.update(nonempty_row)

## Save one CSV per local year

Columns that are empty for the entire requested period are omitted. Each CSV is saved below `data/raw/<year>/`.

In [5]:
output_columns = [name for name in column_order if name in columns_with_data]
output_columns.insert(output_columns.index("DATE") + 1, "DATE_local")

years_to_save = range(START.year, END.year + 1)
for year in tqdm(years_to_save, desc="Saving yearly CSV files", unit="year"):
    first_day = max(START, date(year, 1, 1))
    last_day = min(END, date(year + 1, 1, 1)) - timedelta(days=1)

    year_dir = OUT_DIR / str(year)
    year_dir.mkdir(parents=True, exist_ok=True)
    output_path = year_dir / f"rdu_ghcnh_{first_day}_to_{last_day}.csv"

    with output_path.open("w", newline="", encoding="utf-8") as output_file:
        writer = csv.DictWriter(
            output_file,
            fieldnames=output_columns,
            extrasaction="ignore",
        )
        writer.writeheader()
        writer.writerows(rows_by_year[year])

    relative_path = output_path.relative_to(PROJECT_ROOT)
    print(f"Saved {relative_path} ({len(rows_by_year[year]):,} rows)")

Saving yearly CSV files:   0%|          | 0/12 [00:00<?, ?year/s]

Saved data/raw/2015/rdu_ghcnh_2015-01-01_to_2015-12-31.csv (8,760 rows)
Saved data/raw/2016/rdu_ghcnh_2016-01-01_to_2016-12-31.csv (8,781 rows)
Saved data/raw/2017/rdu_ghcnh_2017-01-01_to_2017-12-31.csv (8,757 rows)
Saved data/raw/2018/rdu_ghcnh_2018-01-01_to_2018-12-31.csv (8,757 rows)
Saved data/raw/2019/rdu_ghcnh_2019-01-01_to_2019-12-31.csv (8,759 rows)
Saved data/raw/2020/rdu_ghcnh_2020-01-01_to_2020-12-31.csv (8,784 rows)
Saved data/raw/2021/rdu_ghcnh_2021-01-01_to_2021-12-31.csv (8,756 rows)
Saved data/raw/2022/rdu_ghcnh_2022-01-01_to_2022-12-31.csv (8,760 rows)
Saved data/raw/2023/rdu_ghcnh_2023-01-01_to_2023-12-31.csv (8,760 rows)
Saved data/raw/2024/rdu_ghcnh_2024-01-01_to_2024-12-31.csv (8,784 rows)
Saved data/raw/2025/rdu_ghcnh_2025-01-01_to_2025-12-31.csv (8,641 rows)
Saved data/raw/2026/rdu_ghcnh_2026-01-01_to_2026-09-16.csv (6,172 rows)


## Merge the yearly CSV files

Combine the files in `data/raw/<year>/` into one CSV in `data/raw/`. The header is written once, and rows are streamed instead of loading the full dataset into memory.

In [6]:
last_date = END - timedelta(days=1)
merged_file = OUT_DIR / f"rdu_ghcnh_{START}_to_{last_date}.csv"

yearly_files = []
for year in range(START.year, END.year + 1):
    matches = sorted((OUT_DIR / str(year)).glob("*.csv"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one CSV in {OUT_DIR / str(year)}, found {len(matches)}"
        )
    yearly_files.append(matches[0])

expected_header = None
total_rows = 0

with merged_file.open("wb") as destination:
    for yearly_file in tqdm(yearly_files, desc="Merging yearly CSV files", unit="file"):
        with yearly_file.open("rb") as source:
            header = source.readline()
            if not header:
                raise ValueError(f"CSV is empty: {yearly_file}")

            if expected_header is None:
                expected_header = header
                destination.write(header)
            elif header != expected_header:
                raise ValueError(f"CSV header does not match: {yearly_file}")

            for line in source:
                destination.write(line)
                total_rows += 1

print(f"Saved {merged_file.relative_to(PROJECT_ROOT)} ({total_rows:,} rows)")

Merging yearly CSV files:   0%|          | 0/12 [00:00<?, ?file/s]

Saved data/raw/rdu_ghcnh_2015-01-01_to_2026-09-16.csv (102,471 rows)


## Explore the merged data

Load the merged CSV with pandas, inspect its columns and first five rows, and count missing values. Columns include measurements, station/time fields, and quality/source metadata; they are not all model inputs. Quality filtering and missing-data treatment belong in the next notebook.

In [7]:
import pandas as pd

data_file = PROJECT_ROOT / "data" / "raw" / (
    f"rdu_ghcnh_{START}_to_{END - timedelta(days=1)}.csv"
)
weather = pd.read_csv(data_file, low_memory=False)

print(f"Rows: {len(weather):,}")
print(f"Columns: {len(weather.columns):,}")

Rows: 102,471
Columns: 226


In [8]:
print(f"Headers ({len(weather.columns)}):")
for number, header in enumerate(weather.columns, start=1):
    print(f"{number:>3}. {header}")

Headers (226):
  1. STATION
  2. Station_name
  3. DATE
  4. DATE_local
  5. Year
  6. Month
  7. Day
  8. Hour
  9. Minute
 10. LATITUDE
 11. LONGITUDE
 12. ELEVATION
 13. temperature
 14. temperature_Quality_Code
 15. temperature_Report_Type
 16. temperature_Source_Code
 17. temperature_Source_Station_ID
 18. dew_point_temperature
 19. dew_point_temperature_Quality_Code
 20. dew_point_temperature_Report_Type
 21. dew_point_temperature_Source_Code
 22. dew_point_temperature_Source_Station_ID
 23. station_level_pressure
 24. station_level_pressure_Quality_Code
 25. station_level_pressure_Report_Type
 26. station_level_pressure_Source_Code
 27. station_level_pressure_Source_Station_ID
 28. sea_level_pressure
 29. sea_level_pressure_Quality_Code
 30. sea_level_pressure_Report_Type
 31. sea_level_pressure_Source_Code
 32. sea_level_pressure_Source_Station_ID
 33. wind_direction
 34. wind_direction_Measurement_Code
 35. wind_direction_Quality_Code
 36. wind_direction_Report_Type
 37. wind_

In [9]:
pd.set_option("display.max_columns", None)
weather.head()

,STATION,Station_name,DATE,DATE_local,Year,Month,Day,Hour,Minute,LATITUDE,LONGITUDE,ELEVATION,temperature,temperature_Quality_Code,temperature_Report_Type,temperature_Source_Code,temperature_Source_Station_ID,dew_point_temperature,dew_point_temperature_Quality_Code,dew_point_temperature_Report_Type,dew_point_temperature_Source_Code,dew_point_temperature_Source_Station_ID,station_level_pressure,station_level_pressure_Quality_Code,station_level_pressure_Report_Type,station_level_pressure_Source_Code,station_level_pressure_Source_Station_ID,sea_level_pressure,sea_level_pressure_Quality_Code,sea_level_pressure_Report_Type,sea_level_pressure_Source_Code,sea_level_pressure_Source_Station_ID,wind_direction,wind_direction_Measurement_Code,wind_direction_Quality_Code,wind_direction_Report_Type,wind_direction_Source_Code,wind_direction_Source_Station_ID,wind_speed,wind_speed_Measurement_Code,wind_speed_Quality_Code,wind_speed_Report_Type,wind_speed_Source_Code,wind_speed_Source_Station_ID,wind_gust,wind_gust_Quality_Code,wind_gust_Report_Type,wind_gust_Source_Code,wind_gust_Source_Station_ID,precipitation,precipitation_Measurement_Code,precipitation_Quality_Code,precipitation_Report_Type,precipitation_Source_Code,precipitation_Source_Station_ID,relative_humidity,relative_humidity_Measurement_Code,relative_humidity_Quality_Code,relative_humidity_Report_Type,relative_humidity_Source_Code,relative_humidity_Source_Station_ID,wet_bulb_temperature,wet_bulb_temperature_Measurement_Code,wet_bulb_temperature_Quality_Code,wet_bulb_temperature_Report_Type,wet_bulb_temperature_Source_Code,wet_bulb_temperature_Source_Station_ID,pres_wx_MW1,pres_wx_MW1_Quality_Code,pres_wx_MW1_Report_Type,pres_wx_MW1_Source_Code,pres_wx_MW1_Source_Station_ID,pres_wx_MW2,pres_wx_MW2_Quality_Code,pres_wx_MW2_Report_Type,pres_wx_MW2_Source_Code,pres_wx_MW2_Source_Station_ID,pres_wx_MW3,pres_wx_MW3_Quality_Code,pres_wx_MW3_Report_Type,pres_wx_MW3_Source_Code,pres_wx_MW3_Source_Station_ID,pres_wx_AU1,pres_wx_AU1_Quality_Code,pres_wx_AU1_Report_Type,pres_wx_AU1_Source_Code,pres_wx_AU1_Source_Station_ID,pres_wx_AU2,pres_wx_AU2_Quality_Code,pres_wx_AU2_Report_Type,pres_wx_AU2_Source_Code,pres_wx_AU2_Source_Station_ID,pres_wx_AU3,pres_wx_AU3_Quality_Code,pres_wx_AU3_Report_Type,pres_wx_AU3_Source_Code,pres_wx_AU3_Source_Station_ID,pres_wx_AW1,pres_wx_AW1_Quality_Code,pres_wx_AW1_Report_Type,pres_wx_AW1_Source_Code,pres_wx_AW1_Source_Station_ID,pres_wx_AW2,pres_wx_AW2_Quality_Code,pres_wx_AW2_Report_Type,pres_wx_AW2_Source_Code,pres_wx_AW2_Source_Station_ID,pres_wx_AW3,pres_wx_AW3_Quality_Code,pres_wx_AW3_Report_Type,pres_wx_AW3_Source_Code,pres_wx_AW3_Source_Station_ID,snow_depth,snow_depth_Quality_Code,snow_depth_Report_Type,snow_depth_Source_Code,snow_depth_Source_Station_ID,visibility,visibility_Measurement_Code,visibility_Quality_Code,visibility_Report_Type,visibility_Source_Code,visibility_Source_Station_ID,altimeter,altimeter_Quality_Code,altimeter_Report_Type,altimeter_Source_Code,altimeter_Source_Station_ID,pressure_3hr_change,pressure_3hr_change_Measurement_Code,pressure_3hr_change_Quality_Code,pressure_3hr_change_Report_Type,pressure_3hr_change_Source_Code,pressure_3hr_change_Source_Station_ID,sky_condition,sky_condition_Quality_Code,sky_condition_Report_Type,sky_condition_Source_Code,sky_condition_Source_Station_ID,sky_condition_baseht,sky_condition_baseht_Measurement_Code,sky_condition_baseht_Quality_Code,sky_condition_baseht_Report_Type,sky_condition_baseht_Source_Code,sky_condition_baseht_Source_Station_ID,ceiling_height,ceiling_height_Measurement_Code,ceiling_height_Quality_Code,ceiling_height_Report_Type,ceiling_height_Source_Code,ceiling_height_Source_Station_ID,sky_cover_layer_1,sky_cover_layer_1_Report_Type,sky_cover_layer_1_Source_Code,sky_cover_layer_1_Source_Station_ID,sky_cover_layer_baseht_1,sky_cover_layer_baseht_1_Measurement_Code,sky_cover_layer_baseht_1_Quality_Code,sky_cover_layer_baseht_1_Report_Type,sky_cover_layer_baseht_1_Source_Code,sky_cover

In [10]:
nan_summary = pd.DataFrame(
    {
        "NaN count": weather.isna().sum(),
        "NaN percent": (weather.isna().mean() * 100).round(2),
    }
).sort_values("NaN count", ascending=False)
nan_summary.index.name = "Feature"

nan_summary

,NaN count,NaN percent
Feature,,
pres_wx_MW3_Quality_Code,102468,100.00
wet_bulb_temperature_Quality_Code,102462,99.99
relative_humidity_Quality_Code,102457,99.99
pres_wx_MW3_Source_Station_ID,102451,99.98
pres_wx_MW3,102451,99.98
...,...,...
relative_humidity_Source_Station_ID,0,0.00
Station_name,0,0.00
altimeter,0,0.00


In [11]:
high_nan_features = (
    nan_summary.loc[nan_summary["NaN percent"] > 50]
    .reset_index()
    .sort_values("NaN percent", ascending=False)
    .reset_index(drop=True)
)

print(f"Features with more than 50% NaN: {len(high_nan_features)}")
high_nan_features

Features with more than 50% NaN: 118


,Feature,NaN count,NaN percent
0,pres_wx_MW3_Quality_Code,102468,100.00
1,relative_humidity_Quality_Code,102457,99.99
2,wet_bulb_temperature_Quality_Code,102462,99.99
3,pres_wx_MW3_Source_Station_ID,102451,99.98
4,pres_wx_MW3,102451,99.98
...,...,...,...
113,sky_condition_Quality_Code,63347,61.82
114,sky_condition_Report_Type,62328,60.83
115,sky_condition_Source_Station_ID,62328,60.83
116,sky_condition,62328,60.83


## Verify the downloaded dataset

Check the station, report type, observation minute, timestamp ordering, cutoff, and yearly/merged row counts and headers. These checks only read existing files and leave `weather` unchanged.

Compare observed timestamps with a complete hourly UTC grid. Missing rows are different from missing values inside existing rows. Repeated offset-free local timestamps are expected when daylight saving time ends; their UTC timestamps must still be unique.

In [12]:
required_columns = {
    "STATION", "DATE", "DATE_local", "Hour", "Minute",
    "temperature", "temperature_Report_Type",
}
assert required_columns.issubset(weather.columns), "Required columns are missing."
assert not weather.empty, "The merged dataset is empty."
assert weather["STATION"].eq(STATION_ID).all(), "Unexpected station."
assert weather["temperature_Report_Type"].eq("FM15").all(), "Unexpected report type."
assert weather["Minute"].eq(51).all(), "Unexpected observation minute."

utc_dates = pd.to_datetime(weather["DATE"], utc=True, errors="raise")
local_dates = utc_dates.dt.tz_convert(NY)
start_utc = pd.Timestamp(START, tz=NY).tz_convert("UTC")
cutoff_utc = pd.Timestamp(END, tz=NY).tz_convert("UTC")

assert utc_dates.notna().all(), "A timestamp is missing."
assert utc_dates.is_unique, "Duplicate UTC observations."
assert utc_dates.is_monotonic_increasing, "Observations are out of order."
assert ((utc_dates >= start_utc) & (utc_dates < cutoff_utc)).all(), "Date outside the requested range."
assert weather["Hour"].eq(utc_dates.dt.hour).all(), "Hour must represent UTC."
assert local_dates.dt.strftime("%Y-%m-%d %H:%M").eq(weather["DATE_local"]).all(), "Local timestamps do not match UTC."

yearly_row_counts = {}
for year in range(START.year, END.year + 1):
    first_day = max(START, date(year, 1, 1))
    last_day = min(END, date(year + 1, 1, 1)) - timedelta(days=1)
    yearly_path = OUT_DIR / str(year) / f"rdu_ghcnh_{first_day}_to_{last_day}.csv"
    assert yearly_path.is_file(), f"Missing yearly CSV: {yearly_path}"
    with yearly_path.open(newline="", encoding="utf-8") as source:
        reader = csv.reader(source)
        assert next(reader) == weather.columns.tolist(), f"Header mismatch: {yearly_path}"
        yearly_row_counts[year] = sum(1 for row in reader if row)
    assert yearly_row_counts[year] == int(local_dates.dt.year.eq(year).sum()), f"Yearly row count mismatch: {year}"

assert sum(yearly_row_counts.values()) == len(weather), "Yearly and merged row totals differ."

expected_utc = pd.date_range(
    start=start_utc + pd.Timedelta(minutes=51),
    end=cutoff_utc, freq="h", inclusive="left",
)
assert utc_dates.isin(expected_utc).all(), "Observation outside the expected hourly grid."
missing_utc_hours = expected_utc.difference(pd.DatetimeIndex(utc_dates))
repeated_local_hours = int(local_dates.dt.tz_localize(None).duplicated().sum())

print(f"All acquisition checks passed for {len(yearly_row_counts)} yearly CSVs.")
print(f"Expected hourly observations: {len(expected_utc):,}")
print(f"Observed hourly records: {len(weather):,}")
print(f"Missing hourly records: {len(missing_utc_hours):,}")
print(f"Repeated offset-free local timestamps: {repeated_local_hours:,}")

All acquisition checks passed for 12 yearly CSVs.
Expected hourly observations: 102,647
Observed hourly records: 102,471
Missing hourly records: 176
Repeated offset-free local timestamps: 11


## Final data-sourcing summary

The table below summarizes the loaded files. Hourly coverage uses the UTC grid so daylight saving changes do not create false gaps. Temperature missingness describes existing rows only.

The files under `data/raw/` contain the selected FM15, minute-51 observations, with globally empty columns omitted. Original measurement values and quality/source metadata are retained for cleaning decisions in the next notebook.

In [13]:
sourcing_summary = pd.Series(
    {
        "Station": STATION_ID,
        "Source": "NOAA NCEI GHCNh",
        "Merged CSV": str(data_file.relative_to(PROJECT_ROOT)),
        "Yearly CSVs": len(yearly_row_counts),
        "First observation (Eastern)": local_dates.min().isoformat(),
        "Last observation (Eastern)": local_dates.max().isoformat(),
        "Exclusive cutoff (Eastern)": pd.Timestamp(END, tz=NY).isoformat(),
        "Rows": len(weather),
        "Columns (measurements + metadata)": len(weather.columns),
        "Temperature unit": "degrees Celsius",
        "Missing temperature values in observed rows": int(weather["temperature"].isna().sum()),
        "Expected hourly records": len(expected_utc),
        "Missing hourly records": len(missing_utc_hours),
        "Hourly coverage": f"{len(weather) / len(expected_utc):.2%}",
        "Duplicate UTC timestamps": int(utc_dates.duplicated().sum()),
        "Repeated offset-free local timestamps": repeated_local_hours,
        "Columns with more than 50% NaN": int((weather.isna().mean() > 0.5).sum()),
    },
    name="Value",
).rename_axis("Metric").to_frame()

sourcing_summary

,Value
Metric,
Station,USW00013722
Source,NOAA NCEI GHCNh
Merged CSV,data/raw/rdu_ghcnh_2015-01-01_to_2026-09-16.csv
Yearly CSVs,12
First observation (Eastern),2015-01-01T00:51:00-05:00
Last observation (Eastern),2026-09-16T23:51:00-04:00
Exclusive cutoff (Eastern),2026-09-17T00:00:00-04:00
Rows,102471
Columns (measurements + metadata),226


## Handoff to cleaning and EDA

The data-sourcing output is `data/raw/rdu_ghcnh_2015-01-01_to_2026-09-16.csv`. Next, `02_cleaning_eda.ipynb` will establish a timezone-aware hourly index, investigate gaps and quality flags, choose cleaning rules, and explore temperature patterns. Cleaned data will be saved separately under `data/processed/`. Feature construction and model training follow those checks.

To inspect existing downloads without rewriting raw files, run the imports/settings cells, the merged-CSV loading cell, and the verification/summary cells above. The download, save, and merge cells regenerate raw files when run.